In [1]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [2]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "test"))


In [3]:
from src.demos.ekg.pydantic_rag import PydanticRag

url = global_config().get_dsn("vector_store.postgres_url", driver="asyncpg")

rag = PydanticRag(
    model_definition=test_schema, postgres_url=url, embeddings_id="qwen3_06b_deepinfra", llm_id=None, kv_store_id="file"
)


2025-08-12 16:32:27.846 | DEBUG    | src.ai_core.llm_factory:get_llm:530 - get LLM:'kimi_k2_openrouter' -extra: {'temperature': 0.0}
2025-08-12 16:32:28.819 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7eae7ab7e480>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-12 16:32:28.858 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7eae7abe3dd0> (PGVectorStore)


2025-08-12 16:32:28.907 | WARNING  | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-12 16:32:28.908 | INFO     | src.ai_core.vector_store_factory:get:230 - get vector store  : PgVector/pydantic_fields_qwen3_06b cache_embeddings: False


In [5]:
# A tiny markdown document

doc_text = """
# Jane Doe
Jane is 29 years old and can be reached at jane.doe@example.com (personal adress) and J.doe@apple.com (professional email).
She lives rue de la Concorde, 31000 Toulouse France.

"""
doc_id = "test"
key = "example Jane Joe #1"
# 1. Analyse → structured Pydantic object

person = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", person)

assert person


2025-08-12 16:32:49.833 | DEBUG    | src.utils.pydantic.kv_store:load_object_from_kvstore:89 - read 'Person/test.json' from KV store


Structured result:
Person(
    name='Jane Doe',
    age=29,
    email=[
        Email(url='jane.doe@example.com', email_type='personal'),
        Email(url='J.doe@apple.com', email_type='professional')
    ],
    address=Address(street='rue de la Concorde', city='Toulouse', zip_code='31000', country='France'),
    doc_id='test'
)

In [ ]:
chunks = rag.chunck(person)


In [ ]:
debug(chunks)

In [ ]:
# 2. Index the document
#rag.store_chunks(chunks)
print("Document stored.")



In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore("e-mail address", k=2, filter={"langchain_metadata ->> description": {"$eq":"Age in years"}})
print("Vector hits:", hits)